In [0]:
from pyspark.sql.functions import (
    col, upper, trim, when, regexp_replace, current_timestamp,
    to_date, to_timestamp, round as spark_round, length, lit,
    coalesce, initcap, lower, regexp_extract, sha2, concat_ws, expr
)
from delta.tables import DeltaTable

CATALOG = "workspace"
SCHEMA  = "default"

def tbl(name):
    return f"{CATALOG}.{SCHEMA}.{name}"

# =============================================================
# SILVER LAYER – Automobile Analytics Platform
# Notebook version: reads from bronze catalog tables,
# writes silver tables back to workspace.default.
#
# _row_hash is NOT present in bronze tables (was added by DLT).
# We compute it here with sha2(concat_ws(...)) on the tracked
# columns so SCD2 MERGE logic works the same way.
#
# _ingest_ts may also be absent from bronze – we use
# current_timestamp() as a fallback.
# =============================================================


# ─────────────────────────────────────────────────────────────
# HELPER: VIN validator
# ─────────────────────────────────────────────────────────────
def is_valid_vin(vin_col):
    return (
        (length(vin_col) == 17) &
        (regexp_extract(vin_col, r"^[A-HJ-NPR-Z0-9]{17}$", 0) != "")
    )

# ─────────────────────────────────────────────────────────────
# HELPER: drop columns only if they exist
# ─────────────────────────────────────────────────────────────
def safe_drop(df, *cols_to_drop):
    existing = set(df.columns)
    return df.drop(*[c for c in cols_to_drop if c in existing])


# =============================================================
# SCD TYPE 2 – CUSTOMER DIMENSION
# _row_hash computed from all tracked attributes
# =============================================================

bronze_customer = (
    spark.read.table(tbl("bronze_customer_new"))
    .filter(col("customer_id").isNotNull())
    .select(
        "customer_id",
        initcap(trim(col("customer_name"))).alias("customer_name"),
        upper(trim(col("gender"))).alias("gender"),
        expr("try_cast(age as int)").alias("age"),
        initcap(trim(col("city"))).alias("city"),
        upper(trim(col("state"))).alias("state"),
        upper(trim(col("income_range"))).alias("income_range"),
        sha2(col("email"), 256).alias("email_hash"),
        regexp_replace(
            col("contact_number"), r"\d(?=\d{4})", "*"
        ).alias("contact_number_masked"),
    )
    # Compute _row_hash from tracked columns (replaces DLT-generated hash)
    .withColumn("_row_hash", sha2(concat_ws("||",
        col("customer_name"), col("gender"), col("age").cast("string"),
        col("city"), col("state"), col("income_range"),
        col("email_hash"), col("contact_number_masked")
    ), 256))
    .withColumn("_ingest_ts", current_timestamp())
)

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {tbl("silver_customer_scd2")} (
        customer_id            STRING,
        customer_name          STRING,
        gender                 STRING,
        age                    INT,
        city                   STRING,
        state                  STRING,
        income_range           STRING,
        email_hash             STRING,
        contact_number_masked  STRING,
        _row_hash              STRING,
        _ingest_ts             TIMESTAMP,
        scd_start_date         TIMESTAMP,
        scd_end_date           TIMESTAMP,
        is_current             BOOLEAN
    )
    USING DELTA
""")

target = DeltaTable.forName(spark, tbl("silver_customer_scd2"))

# Step 1: Close changed records
target.alias("tgt").merge(
    bronze_customer.alias("src"),
    "tgt.customer_id = src.customer_id AND tgt.is_current = true AND tgt._row_hash != src._row_hash"
).whenMatchedUpdate(set={
    "is_current":   lit(False),
    "scd_end_date": current_timestamp()
}).execute()

# Step 2: Insert new / changed records (skip rows whose hash already exists as current)
(
    bronze_customer.alias("src")
    .join(
        spark.read.table(tbl("silver_customer_scd2"))
            .filter(col("is_current") == True)
            .select("customer_id", "_row_hash")
            .alias("tgt"),
        (col("src.customer_id") == col("tgt.customer_id")) &
        (col("src._row_hash")   == col("tgt._row_hash")),
        "left_anti"
    )
    .withColumn("scd_start_date", current_timestamp())
    .withColumn("scd_end_date",   lit(None).cast("timestamp"))
    .withColumn("is_current",     lit(True))
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(tbl("silver_customer_scd2"))
)

print("✅ silver_customer_scd2 updated")


# =============================================================
# SCD TYPE 2 – DEALER DIMENSION
# =============================================================

bronze_dealer = (
    spark.read.table(tbl("bronze_dealer_new"))
    .filter(col("dealer_id").isNotNull())
    .select(
        "dealer_id",
        initcap(trim(col("dealer_name"))).alias("dealer_name"),
        upper(trim(col("dealer_type"))).alias("dealer_type"),
        initcap(trim(col("city"))).alias("city"),
        upper(trim(col("state"))).alias("state"),
        upper(trim(col("region"))).alias("region"),
        to_date(col("opening_date")).alias("opening_date"),
        expr("try_cast(rating as double)").alias("rating"),
        expr("try_cast(capacity as int)").alias("capacity"),
        regexp_replace(
            col("contact_number"), r"\d(?=\d{4})", "*"
        ).alias("contact_number_masked"),
    )
    .withColumn("_row_hash", sha2(concat_ws("||",
        col("dealer_name"), col("dealer_type"), col("city"), col("state"),
        col("region"), col("rating").cast("string"),
        col("capacity").cast("string"), col("contact_number_masked")
    ), 256))
    .withColumn("_ingest_ts", current_timestamp())
)

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {tbl("silver_dealer_scd2")} (
        dealer_id              STRING,
        dealer_name            STRING,
        dealer_type            STRING,
        city                   STRING,
        state                  STRING,
        region                 STRING,
        opening_date           DATE,
        rating                 DOUBLE,
        capacity               INT,
        contact_number_masked  STRING,
        _row_hash              STRING,
        _ingest_ts             TIMESTAMP,
        scd_start_date         TIMESTAMP,
        scd_end_date           TIMESTAMP,
        is_current             BOOLEAN
    )
    USING DELTA
""")

target = DeltaTable.forName(spark, tbl("silver_dealer_scd2"))

target.alias("tgt").merge(
    bronze_dealer.alias("src"),
    "tgt.dealer_id = src.dealer_id AND tgt.is_current = true AND tgt._row_hash != src._row_hash"
).whenMatchedUpdate(set={
    "is_current":   lit(False),
    "scd_end_date": current_timestamp()
}).execute()

(
    bronze_dealer.alias("src")
    .join(
        spark.read.table(tbl("silver_dealer_scd2"))
            .filter(col("is_current") == True)
            .select("dealer_id", "_row_hash")
            .alias("tgt"),
        (col("src.dealer_id") == col("tgt.dealer_id")) &
        (col("src._row_hash") == col("tgt._row_hash")),
        "left_anti"
    )
    .withColumn("scd_start_date", current_timestamp())
    .withColumn("scd_end_date",   lit(None).cast("timestamp"))
    .withColumn("is_current",     lit(True))
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(tbl("silver_dealer_scd2"))
)

print("✅ silver_dealer_scd2 updated")


# =============================================================
# SCD TYPE 2 – VEHICLE MASTER DIMENSION
# =============================================================

bronze_vehicle = (
    spark.read.table(tbl("bronze_vehicle_master_new"))
    .filter(col("vin").isNotNull())
    .filter(length(col("vin")) == 17)
    .select(
        "vin",
        "model_code",
        initcap(trim(col("model_name"))).alias("model_name"),
        upper(trim(col("variant"))).alias("variant"),
        upper(trim(col("fuel_type"))).alias("fuel_type"),
        upper(trim(col("transmission"))).alias("transmission"),
        expr("try_cast(engine_capacity as int)").alias("engine_capacity"),
        to_date(col("manufacture_date")).alias("manufacture_date"),
        upper(trim(col("plant_id"))).alias("plant_id"),
        initcap(trim(col("color"))).alias("color"),
        spark_round(expr("try_cast(ex_showroom_price as double)"), 2)
            .alias("ex_showroom_price"),
    )
    .withColumn("_row_hash", sha2(concat_ws("||",
        col("model_name"), col("variant"), col("fuel_type"),
        col("transmission"), col("engine_capacity").cast("string"),
        col("color"), col("ex_showroom_price").cast("string")
    ), 256))
    .withColumn("_ingest_ts", current_timestamp())
)

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {tbl("silver_vehicle_scd2")} (
        vin                STRING,
        model_code         STRING,
        model_name         STRING,
        variant            STRING,
        fuel_type          STRING,
        transmission       STRING,
        engine_capacity    INT,
        manufacture_date   DATE,
        plant_id           STRING,
        color              STRING,
        ex_showroom_price  DOUBLE,
        _row_hash          STRING,
        _ingest_ts         TIMESTAMP,
        scd_start_date     TIMESTAMP,
        scd_end_date       TIMESTAMP,
        is_current         BOOLEAN
    )
    USING DELTA
""")

target = DeltaTable.forName(spark, tbl("silver_vehicle_scd2"))

target.alias("tgt").merge(
    bronze_vehicle.alias("src"),
    "tgt.vin = src.vin AND tgt.is_current = true AND tgt._row_hash != src._row_hash"
).whenMatchedUpdate(set={
    "is_current":   lit(False),
    "scd_end_date": current_timestamp()
}).execute()

(
    bronze_vehicle.alias("src")
    .join(
        spark.read.table(tbl("silver_vehicle_scd2"))
            .filter(col("is_current") == True)
            .select("vin", "_row_hash")
            .alias("tgt"),
        (col("src.vin")       == col("tgt.vin")) &
        (col("src._row_hash") == col("tgt._row_hash")),
        "left_anti"
    )
    .withColumn("scd_start_date", current_timestamp())
    .withColumn("scd_end_date",   lit(None).cast("timestamp"))
    .withColumn("is_current",     lit(True))
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(tbl("silver_vehicle_scd2"))
)

print("✅ silver_vehicle_scd2 updated")


# =============================================================
# REUSABLE CURRENT-RECORD VIEWS
# =============================================================

customer_current = (
    spark.read.table(tbl("silver_customer_scd2"))
    .filter(col("is_current") == True)
)

dealer_current = (
    spark.read.table(tbl("silver_dealer_scd2"))
    .filter(col("is_current") == True)
)

vehicle_current = (
    spark.read.table(tbl("silver_vehicle_scd2"))
    .filter(col("is_current") == True)
)


# =============================================================
# 1. SILVER SALES
# =============================================================

sales    = spark.read.table(tbl("bronze_sales_new"))
customer = customer_current
dealer   = dealer_current
vehicle  = vehicle_current

customer_joined = customer.select(
    "customer_id", "customer_name", "gender", "age",
    col("city").alias("customer_city"),
    col("state").alias("customer_state"),
    "income_range", "email_hash", "contact_number_masked"
)

dealer_joined = dealer.select(
    "dealer_id",
    col("dealer_name"),
    col("dealer_type"),
    col("region").alias("dealer_region"),
    col("city").alias("dealer_city"),
    col("state").alias("dealer_state"),
    col("rating").alias("dealer_rating")
)

vehicle_joined = vehicle.select(
    "vin", "model_code", "model_name",
    "fuel_type", "transmission", "variant", "ex_showroom_price"
)

silver_sales = (
    sales
    .filter(col("vin").isNotNull())
    .filter(expr("try_cast(sale_amount as double)") > 0)
    .filter(is_valid_vin(col("vin")))
    .dropDuplicates(["vin", "sale_date"])

    .withColumn("sale_date",    to_date(col("sale_date")))
    .withColumn("region",       upper(trim(col("region"))))
    .withColumn("channel",      upper(trim(col("channel"))))
    .withColumn("payment_mode", upper(trim(col("payment_mode"))))
    .withColumn("city",         initcap(trim(col("city"))))
    .withColumn("sale_amount",  spark_round(expr("try_cast(sale_amount as double)"), 2))
    .withColumn("discount",     spark_round(coalesce(expr("try_cast(discount as double)"), lit(0.0)), 2))
    .withColumn("tax_amount",   spark_round(coalesce(expr("try_cast(tax_amount as double)"), lit(0.0)), 2))
    .withColumn("net_revenue",  spark_round(
        col("sale_amount") - col("discount") + col("tax_amount"), 2))

    .join(customer_joined, "customer_id", "left")
    .join(dealer_joined,   "dealer_id",   "left")
    .join(vehicle_joined,  "vin",         "left")

    .withColumn("_processed_ts", current_timestamp())
    .transform(lambda df: safe_drop(df, "email", "contact_number",
                                        "_ingest_ts", "sale_date_ts", "_row_hash"))
)

spark.sql(f"DROP TABLE IF EXISTS {tbl('silver_sales_new')}")
(
    silver_sales.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tbl("silver_sales_new"))
)
print("✅ silver_sales_new written")


# =============================================================
# 2. SILVER PRODUCTION
# =============================================================

production    = spark.read.table(tbl("bronze_production_new"))
vehicle_clean = vehicle_current.select(
    "vin", "model_code", "model_name", "fuel_type", "variant", "plant_id"
)

silver_production = (
    production
    .filter(col("vin").isNotNull())
    .filter(expr("try_cast(production_time_minutes as int)") > 0)
    .dropDuplicates(["production_id"])

    .withColumn("production_date", to_date(col("production_date")))
    .withColumn("status",          upper(trim(col("status"))))
    .withColumn("shift",           upper(trim(col("shift"))))
    .withColumn("assembly_line",   upper(trim(col("assembly_line"))))
    .withColumn("plant_id",        upper(trim(col("plant_id"))))

    .withColumn("is_completed",
        when(col("status") == "COMPLETED", 1).otherwise(0))
    .withColumn("is_delayed",
        when(col("status") == "DELAYED", 1).otherwise(0))

    .join(vehicle_clean.drop("plant_id"), "vin", "left")

    .withColumn("_processed_ts", current_timestamp())
    .transform(lambda df: safe_drop(df, "_ingest_ts", "production_date_ts", "_row_hash"))
)

spark.sql(f"DROP TABLE IF EXISTS {tbl('silver_production_new')}")
(
    silver_production.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tbl("silver_production_new"))
)
print("✅ silver_production_new written")


# =============================================================
# 3. SILVER SERVICE
# =============================================================

service  = spark.read.table(tbl("bronze_service_new"))
warranty = spark.read.table(tbl("bronze_warranty_new"))

warranty_clean = warranty.select(
    "vin",
    col("claim_id"),
    to_date(col("claim_date")).alias("claim_date"),
    upper(trim(col("claim_status"))).alias("claim_status"),
    spark_round(expr("try_cast(claim_amount as double)"), 2).alias("claim_amount"),
    "part_id"
)

dealer_clean = dealer_current.select(
    "dealer_id",
    col("dealer_name"),
    col("region").alias("dealer_region"),
    col("dealer_type")
)

vehicle_svc = vehicle_current.select(
    "vin", "model_code", "model_name", "fuel_type"
)

silver_service = (
    service
    .filter(col("vin").isNotNull())
    .filter(expr("try_cast(service_cost as double)") >= 0)
    .dropDuplicates(["service_id"])

    .withColumn("service_date",  to_date(col("service_date")))
    .withColumn("service_type",  upper(trim(col("service_type"))))
    .withColumn("service_cost",  spark_round(expr("try_cast(service_cost as double)"), 2))
    .withColumn("mileage",       coalesce(col("mileage"), lit(0)))
    .withColumn("customer_feedback_rating",
        when(col("customer_feedback_rating").between(1, 5),
             col("customer_feedback_rating"))
        .otherwise(lit(None)))

    .join(warranty_clean, "vin",       "left")
    .join(dealer_clean,   "dealer_id", "left")
    .join(vehicle_svc,    "vin",       "left")

    .withColumn("is_warranty_claim",
        when(col("claim_id").isNotNull(), 1).otherwise(0))

    .withColumn("_processed_ts", current_timestamp())
    .transform(lambda df: safe_drop(df, "_ingest_ts", "service_date_ts", "_row_hash"))
)

spark.sql(f"DROP TABLE IF EXISTS {tbl('silver_service_new')}")
(
    silver_service.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tbl("silver_service_new"))
)
print("✅ silver_service_new written")


# =============================================================
# 4. SILVER INVENTORY
# =============================================================

inventory    = spark.read.table(tbl("bronze_inventory_new"))
parts        = spark.read.table(tbl("bronze_parts_new"))
dealer_parts = spark.read.table(tbl("bronze_dealer_parts_new"))

parts_clean = parts.select(
    "part_id",
    initcap(trim(col("part_name"))).alias("part_name"),
    upper(trim(col("category"))).alias("category"),
    "supplier_id",
    spark_round(expr("try_cast(unit_cost as double)"), 2).alias("unit_cost"),
    to_date(col("manufacture_date")).alias("manufacture_date"),
    to_date(col("expiry_date")).alias("expiry_date"),
    col("quality_rating")
)

dealer_parts_clean = dealer_parts.select(
    "part_id",
    "dealer_id",
    coalesce(expr("try_cast(available_stock as int)"), lit(0)).alias("dealer_available_stock"),
    to_date(col("last_restock_date")).alias("last_restock_date")
)

silver_inventory = (
    inventory
    .filter(col("part_id").isNotNull())
    .dropDuplicates(["inventory_id"])

    .withColumn("stock_quantity",
        coalesce(expr("try_cast(stock_quantity as int)"), lit(0)))
    .withColumn("reorder_level",
        coalesce(expr("try_cast(reorder_level as int)"), lit(0)))
    .withColumn("warehouse_id",  upper(trim(col("warehouse_id"))))

    .withColumn("is_below_reorder",
        when(col("stock_quantity") < col("reorder_level"), 1).otherwise(0))

    .join(parts_clean,        "part_id", "left")
    .join(dealer_parts_clean, "part_id", "left")

    .withColumn("inventory_value",
        spark_round(col("stock_quantity") * col("unit_cost"), 2))

    .withColumn("_processed_ts", current_timestamp())
    .transform(lambda df: safe_drop(df, "_ingest_ts", "_row_hash"))
)

spark.sql(f"DROP TABLE IF EXISTS {tbl('silver_inventory_new')}")
(
    silver_inventory.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tbl("silver_inventory_new"))
)
print("✅ silver_inventory_new written")


# =============================================================
# 5. SILVER WARRANTY
# =============================================================

warranty_raw = spark.read.table(tbl("bronze_warranty_new"))

parts_wt = parts.select(
    "part_id",
    initcap(trim(col("part_name"))).alias("part_name"),
    upper(trim(col("category"))).alias("part_category"),
    col("quality_rating")
)

vehicle_wt = vehicle_current.select(
    "vin", "model_code", "model_name", "fuel_type"
)

dealer_wt = dealer_current.select(
    "dealer_id",
    col("dealer_name"),
    col("region").alias("dealer_region")
)

silver_warranty = (
    warranty_raw
    .filter(col("vin").isNotNull())
    .filter(expr("try_cast(claim_amount as double)") >= 0)
    .dropDuplicates(["claim_id"])

    .withColumn("claim_date",   to_date(col("claim_date")))
    .withColumn("claim_status", upper(trim(col("claim_status"))))
    .withColumn("claim_amount", spark_round(expr("try_cast(claim_amount as double)"), 2))

    .withColumn("is_approved",
        when(col("claim_status") == "APPROVED", 1).otherwise(0))
    .withColumn("is_rejected",
        when(col("claim_status") == "REJECTED", 1).otherwise(0))

    .join(parts_wt,   "part_id",   "left")
    .join(vehicle_wt, "vin",       "left")
    .join(dealer_wt,  "dealer_id", "left")

    .withColumn("_processed_ts", current_timestamp())
    .transform(lambda df: safe_drop(df, "_ingest_ts", "claim_date_ts", "_row_hash"))
)

spark.sql(f"DROP TABLE IF EXISTS {tbl('silver_warranty_new')}")
(
    silver_warranty.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tbl("silver_warranty_new"))
)
print("✅ silver_warranty_new written")

print("\n🎉 All silver tables updated successfully.")